In [4]:
import langchain
import os
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

EXAMPLE 1  Simple LLM CALL WITH STREAMING



In [8]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage

In [35]:
model = init_chat_model(
    "groq:openai/gpt-oss-120b"
)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000255A70B7C50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000255A70B6490>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [37]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile"
)

response = model.invoke("Hello")
print(response.content)

Hello. How can I help you today?


In [38]:
## Create messages
messages=[
    SystemMessage("You are a helpful AI assistant"),
    HumanMessage("What are the top 2 benefits of using Langchain?")
]



In [39]:
import requests

response = requests.get("https://api.groq.com")
print(response.status_code)

200


In [40]:
## invoke the model
responses=model.invoke(messages)
responses

AIMessage(content="Langchain is a powerful tool that enables developers to build applications powered by large language models (LLMs). The top 2 benefits of using Langchain are:\n\n1. **Streamlined LLM Integration**: Langchain provides a simple and unified API for interacting with various LLMs, making it easier to integrate these models into applications. This allows developers to focus on building their application's core functionality rather than worrying about the complexities of LLM integration.\n\n2. **Modular and Customizable Architecture**: Langchain's modular design enables developers to easily swap out different LLMs, fine-tune models, and experiment with various configurations. This flexibility allows developers to customize their application's language capabilities to meet specific requirements, improving overall performance and efficiency.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 53, 'total_tokens': 203, 'completi

Dynamic Prompt Templates

In [41]:
from langchain_core.prompts import ChatPromptTemplate

## create translation app

translation_template=ChatPromptTemplate.from_messages([
    ("system","You are a professional translator.Translate the follow text {text} from {source_language} to {target_language}. MAintain the tone and style"),
    ("user","{text}")
])

## using the template
prompt=translation_template.invoke({
    "source_language":"English",
    "target_language":"Spanish",
    "text":"Langchain makes building AI application incredibly easy!"
})

In [42]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator.Translate the follow text Langchain makes building AI application incredibly easy! from English to Spanish. MAintain the tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='Langchain makes building AI application incredibly easy!', additional_kwargs={}, response_metadata={})])

In [43]:
translated_response=model.invoke(prompt)
print(translated_response.content)

Langchain hace que construir aplicaciones de inteligencia artificial sea increíblemente fácil.


Building You First Chain

In [49]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Create a more complex chain
def create_story_chain():
    # Template for story generation
    story_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Write a short, engaging story based on the given theme."),
        ("user", "Theme: {theme}\nMain character: {character}\nSetting: {setting}")
    ])
    
    # Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a literary critic. Analyze the following story and provide insights."),
        ("user", "{story}")
    ])
    
    # Build the chain - Method 1: Sequential execution
    story_chain = (
        story_prompt 
        | model 
        | StrOutputParser()
    )
    
    # Create a function to pass the story to analysis
    def analyze_story(story_text):
        return {"story": story_text}
    
    analysis_chain = (
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        | StrOutputParser()
    )
    return analysis_chain

In [50]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short, engaging story based on the given theme.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\nMain character: {character}\nSetting: {setting}'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000255A5E2B230>, async_client=<groq.resources.chat.comple

In [51]:
result = chain.invoke({
    "theme": "artificial intelligence",
    "character": "a curious robot",
    "setting": "a futuristic city"
})

print("Story and Analysis:")
print(result)

Story and Analysis:
This story is a thought-provoking exploration of artificial intelligence, sentience, and the potential consequences of creating conscious beings. On the surface, it appears to be a straightforward narrative about a curious robot named Zeta, but upon closer analysis, it reveals a complex web of themes and ideas.

One of the primary concerns of the story is the nature of consciousness and what it means to be sentient. The author raises important questions about the boundaries between human and artificial intelligence, and whether it is possible for a machine to truly experience emotions, self-awareness, and creativity. The character of Zeta, with its insatiable thirst for knowledge and exploration, serves as a catalyst for this inquiry, and its interactions with Echo-12 provide a framework for understanding the possibilities and implications of artificial sentience.

The concept of Echo-12, an experimental AI designed to push the boundaries of artificial intelligence,